# Inspect Contrastive Privacy Results

This notebook loads a run folder produced by `resolution-analysis` or `text_resolution_analysis`, reuses the cached JSON analysis bundle when available, generates the HTML analysis page, and shows example pairs inline.

Update `OUTPUT_FOLDER` in the next cell to inspect a different run.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUTPUT_FOLDER = REPO_ROOT / 'runs' / 'dicaprio_simple_cp0'
THRESHOLD = 0.0
TOP_N = 5
COMPUTE_SIMILARITY = True
DEVICE = None
BATCH_SIZE = 2
IMAGE_MODEL = None
TEXT_EMBEDDER = None
TEXT_EMBEDDER_MODEL = None
TEXT_EMBEDDER_QUANTIZATION = None
OUTPUT_FOLDER

In [ ]:
from contrastive_privacy.reporting import generate_analysis_artifacts

artifacts = generate_analysis_artifacts(
    OUTPUT_FOLDER,
    title=f'Notebook View: {OUTPUT_FOLDER.name}',
    threshold=THRESHOLD,
    top_n=TOP_N,
    compute_similarity=COMPUTE_SIMILARITY,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    image_model=IMAGE_MODEL,
    text_embedder=TEXT_EMBEDDER,
    text_embedder_model=TEXT_EMBEDDER_MODEL,
    text_embedder_quantization=TEXT_EMBEDDER_QUANTIZATION,
    html_output=OUTPUT_FOLDER / 'analysis_report.notebook.html',
    json_output=OUTPUT_FOLDER / 'analysis_report.notebook.json',
)

bundle = artifacts['bundle']
report_path = artifacts['html_path']
json_path = artifacts['json_path']
(report_path, json_path, artifacts['used_cache'])

In [ ]:
from pprint import pprint

print('Resolution summary')
pprint(bundle['resolution_summary'])
print('\nSimilarity summary')
pprint(bundle['similarity_summary'])

In [ ]:
from IPython.display import HTML, display
from urllib.parse import quote

def _render_image_example(example):
    tiles = []
    for label, key in [
        ('Reference original', 'u_path'),
        ('Reference obfuscated', 'u_obfuscated_path'),
        ('Peer original', 'v_path'),
        ('Peer obfuscated', 'v_obfuscated_path'),
    ]:
        path = example.get(key)
        if path and Path(path).is_file():
            body = f"<img src='{Path(path).resolve().as_uri()}' alt='{label}'>"
        else:
            body = "<div class='missing'>Missing artifact</div>"
        tiles.append(
            f"<figure class='tile'>{body}<figcaption>{label}</figcaption></figure>"
        )
    return (
        "<section class='card'>"
        f"<h3>{example['u_path'].name}</h3>"
        f"<p><strong>Resolution:</strong> {example['resolution']:+.4f}<br>{example['explanation']}</p>"
        f"<div class='grid'>" + ''.join(tiles) + "</div></section>"
    )

def _render_text_example(example):
    blocks = []
    for label, key in [
        ('Reference original', 'u_path'),
        ('Reference obfuscated', 'u_obfuscated_path'),
        ('Peer original', 'v_path'),
        ('Peer obfuscated', 'v_obfuscated_path'),
    ]:
        path = example.get(key)
        text = Path(path).read_text(encoding='utf-8') if path and Path(path).is_file() else 'Missing artifact'
        blocks.append(
            f"<div class='text-block'><div class='label'>{label}</div><pre>{text[:1200]}</pre></div>"
        )
    return (
        "<section class='card'>"
        f"<h3>{example['u_path'].name}</h3>"
        f"<p><strong>Resolution:</strong> {example['resolution']:+.4f}<br>{example['explanation']}</p>"
        f"<div class='grid'>" + ''.join(blocks) + "</div></section>"
    )

def show_examples(section='worst', count=3):
    examples = bundle['examples'][section][:count]
    renderer = _render_image_example if bundle['content_type'] == 'image' else _render_text_example
    css = '''
    <style>
      .card { background: #fffdf8; border: 1px solid #d9cfbf; border-radius: 20px; padding: 18px; margin: 18px 0; }
      .grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 12px; }
      .tile, .text-block { margin: 0; border: 1px solid #d9cfbf; border-radius: 14px; overflow: hidden; background: #f5efe5; }
      .tile img { display: block; width: 100%; height: 260px; object-fit: cover; }
      .tile figcaption, .label { padding: 10px 12px; font: 12px/1.4 sans-serif; color: #665f57; border-top: 1px solid #d9cfbf; }
      .missing { min-height: 260px; display: grid; place-items: center; color: #665f57; }
      pre { margin: 0; padding: 12px; white-space: pre-wrap; font-size: 12px; line-height: 1.5; }
      @media (max-width: 900px) { .grid { grid-template-columns: 1fr; } }
    </style>
    '''
    display(HTML(css + ''.join(renderer(example) for example in examples)))

show_examples('worst', count=3)

In [ ]:
from IPython.display import HTML, display

display(HTML(report_path.read_text(encoding='utf-8')))